# leodragkit aero tutorial: building and plotting a spacecraft

A running example that grows with each aero task: every task adds a section showing what its code produces. Tests check the numbers; this notebook shows the result.

**Before running.** Activate the `leodragkit` environment and install the optional plotting library: `conda install -c conda-forge plotly`. The figures are interactive: drag to rotate, scroll to zoom, click a legend entry to hide it.

**In git.** Clear all outputs before committing (in Jupyter or VS Code: *Clear All Outputs*), so the repository holds code and text, not large figures.

| Section | Task | What it shows |
| --- | --- | --- |
| 1 | A3a | one triangle: corner order sets the outward normal |
| 2 | A3a | a cube built by hand as a `PanelSet` |
| 3 | A3p | the cube in 3-D, coloured by component and by material |
| 4 | A3a, A3p | a macro model: panels without positions |
| 5 | A3b1 | primitives (box, plate, cylinder) at two resolutions, placed by poses |
| 6 | A3b2 | a 3U CubeSat joined from components |
| 7 | A3b3 | orientation checks: the summary table, a broken box, a warning |
| 8 | A3c | placing parts: planes and rotations |
| 9 | A3c | a spacecraft from a YAML file, with its materials |


In [1]:
# %load_ext autoreload
# %autoreload 2

import plotly.io as pio

pio.renderers.default = "notebook_connected"

In [2]:
import numpy as np

from leodragkit.aero.geometry import (
    compute_triangle_properties,
    panel_set_from_panels,
    panel_set_from_triangles,
)
from leodragkit.plots.spacecraft import plot_panel_set_3d

## 1. One triangle (A3a)

A triangle's outward normal follows its corner order: counter-clockwise when seen from outside. These corners go counter-clockwise seen from +z, so the normal points along +z. The area is half the length of the cross product of two edges.

In [3]:
triangle_m = np.array([[[0.0, 0.0, 0.0], [1.0, 0.0, 0.0], [0.0, 1.0, 0.0]]])
properties = compute_triangle_properties(triangle_m)
print("normal:  ", properties.outward_normals[0])
print("area:    ", properties.areas_m2[0], "m²")
print("centroid:", properties.centroids_m[0], "m")

normal:   [0. 0. 1.]
area:     0.5 m²
centroid: [0.33333333 0.33333333 0.        ] m


Listing the same corners in the opposite order flips the normal: the panel now faces −z.

In [4]:
reversed_properties = compute_triangle_properties(triangle_m[:, ::-1, :])
print("normal with the corners reversed:", reversed_properties.outward_normals[0])

normal with the corners reversed: [ 0.  0. -1.]


## 2. A cube built by hand (A3a)

A unit cube centred on the body origin: six faces, two triangles each, every face counter-clockwise seen from outside. Task A3b will replace this by-hand construction with primitives (`box`, `plate`, `cylinder`); writing it out once shows what those primitives produce.

The four side faces are the component `body` and the two z faces are `end_caps`; the ±y faces are `solar_cell` material and the rest `aluminium`.

In [5]:
corners_m = np.array([[x, y, z] for x in (-0.5, 0.5) for y in (-0.5, 0.5) for z in (-0.5, 0.5)])
faces = [
    (0, 1, 3, 2),
    (4, 6, 7, 5),
    (0, 4, 5, 1),
    (2, 3, 7, 6),
    (0, 2, 6, 4),
    (1, 5, 7, 3),
]  # −x +x −y +y −z +z
cube_triangles_m = np.array(
    [corners_m[list(triangle)] for a, b, c, d in faces for triangle in ((a, b, c), (a, c, d))]
)

cube = panel_set_from_triangles(
    cube_triangles_m,
    component_ids=[0] * 8 + [1] * 4,
    material_ids=[0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0],
    component_names=("body", "end_caps"),
    material_names=("aluminium", "solar_cell"),
    source="unit cube, built by hand",
)
print(cube.panel_count, "panels; total area", cube.areas_m2.sum(), "m²")
print("has positions:", cube.has_positions, "| has triangles:", cube.has_triangles)

12 panels; total area 6.0 m²
has positions: True | has triangles: True


## 3. The cube in 3-D (A3p)

Each component is one trace, so the legend lists them. The lines are the outward normals, one per triangle, starting at its centroid; the red, green and blue lines are the body x, y and z axes. Every normal should point away from the cube: one pointing inward would mean a triangle with its corners in the wrong order.

In [6]:
figure = plot_panel_set_3d(cube)
figure

The same cube coloured by material: the ±y faces are solar cells.

In [7]:
plot_panel_set_3d(cube, color_by="material")

## 4. A macro model: panels without positions (A3a, A3p)

Published macro models list each panel's outward normal, area and material, often with no position. Here is a simple one for a 3U CubeSat (0.1 × 0.1 × 0.34 m) with one deployed solar panel (0.3 × 0.1 m), listed once for each side. The normals are rounded to four decimals, as tables often are; `panel_set_from_panels` normalises them.

With no positions there is no shape to draw. The plot shows each normal from the origin, with length proportional to the panel's area, and the title says so. A set like this supports forces, but not shadowing or moments.

In [8]:
normals = [
    [1, 0, 0],
    [-1, 0, 0],
    [0, 1, 0],
    [0, -1, 0],
    [0, 0, 1],
    [0, 0, -1],  # bus faces
    [0.7071, 0.7071, 0],
    [-0.7071, -0.7071, 0],  # one deployed panel, front and back, rounded
]
areas_m2 = [0.034, 0.034, 0.034, 0.034, 0.01, 0.01, 0.03, 0.03]
macro_model = panel_set_from_panels(
    normals,
    areas_m2,
    component_ids=[0] * 6 + [1] * 2,
    material_ids=[0] * 6 + [1] * 2,
    component_names=("bus", "deployed_panel"),
    material_names=("aluminium", "solar_cell"),
    source="3U CubeSat macro model (illustrative)",
)
plot_panel_set_3d(macro_model)

## 5. Primitives and poses (A3b1)

Plates, boxes and cylinders are built centred on their own origin, then placed in the body frame by a pose: a centre plus the component's own x and z axes (y = z × x, so the axes are always right-handed). `max_edge_length_m` sets the resolution; `None` gives the coarsest mesh. The triangle edges are drawn so the resolution is visible.

Until joining arrives in A3b2, a small helper turns one primitive's triangles into a `PanelSet` with a single component and material.

In [9]:
from leodragkit.aero.geometry_builder import (
    build_box_triangles,
    build_cylinder_triangles,
    build_plate_triangles,
    make_pose,
    place_triangles,
)

In [10]:
def single_component(triangles_m, name, material="aluminium"):
    """A PanelSet of one component and one material, for looking at a single primitive."""
    count = len(triangles_m)
    return panel_set_from_triangles(triangles_m, [0] * count, [0] * count, (name,), (material,), source=name)

A 3U CubeSat bus (0.1 × 0.1 × 0.34 m) at the coarsest resolution, then with no triangle edge longer than 2 cm. The force on the box is the same either way; the finer mesh only matters for shadowing.

In [11]:
bus_coarse = single_component(build_box_triangles(0.1, 0.1, 0.34), "bus, coarsest")
bus_fine = single_component(build_box_triangles(0.1, 0.1, 0.34, max_edge_length_m=0.02), "bus, 2 cm")
print(bus_coarse.panel_count, "triangles coarse;", bus_fine.panel_count, "triangles at 2 cm")
plot_panel_set_3d(bus_fine)

12 triangles coarse; 780 triangles at 2 cm


In [12]:
import sys

import numpy
import plotly

print(sys.executable, numpy.__version__, plotly.__version__)

c:\Users\murph\miniconda3\envs\leodragkit\python.exe 2.2.6 7.1.0


A deployed solar panel (0.3 × 0.1 m, both sides), placed beside the bus and tilted by 45° about the body x axis: its own x stays along body x, and its normal (own z) turns from +z towards +y. The axes are typed rounded, as they would be in a YAML file; `make_pose` makes them exact.

In [13]:
panel_pose = make_pose(centre_m=[0.0, 0.2, 0.0], x_axis=[1, 0, 0], z_axis=[0, 0.7071, 0.7071])
panel_m = place_triangles(build_plate_triangles(0.3, 0.1, max_edge_length_m=0.05, two_sided=True), panel_pose)
plot_panel_set_3d(single_component(panel_m, "deployed panel", "solar_cell"), color_by="material")

A capped cylinder, such as a boom or an antenna mast: the side is faceted into segments (at least eight), and the caps are fans from their centres. Every normal should point away from the axis or along it.

In [14]:
boom_m = build_cylinder_triangles(0.02, 0.3, max_edge_length_m=0.02)
plot_panel_set_3d(single_component(boom_m, "boom"))

## 6. A spacecraft from components (A3b2)

A `Component` is one named part with one material, already placed in the body frame. `join_components` turns a list of them into one `PanelSet`: every triangle keeps its component and material, components of the same material share a material id, and the result is validated like any other `PanelSet`.

Here is a 3U CubeSat: an aluminium bus, long axis along z, with two solar panels deployed sideways from its top end, both facing +x and exposed on both sides, and a thin antenna along −z.

In [15]:
from leodragkit.aero.geometry_builder import Component, join_components

resolution_m = 0.02
bus = Component("bus", "aluminium", build_box_triangles(0.1, 0.1, 0.34, max_edge_length_m=resolution_m))


def deployed_panel(name, side):
    """A 0.3 × 0.1 m two-sided panel beside the bus on the ±y side (side = +1 or −1), facing +x."""
    pose = make_pose(centre_m=[0.0, side * (0.05 + 0.15), 0.12], x_axis=[0, side, 0], z_axis=[1, 0, 0])
    plate = build_plate_triangles(0.3, 0.1, max_edge_length_m=resolution_m, two_sided=True)
    return Component(name, "solar_cell", place_triangles(plate, pose))


antenna_pose = make_pose(centre_m=[0.0, 0.0, -0.27], x_axis=[1, 0, 0], z_axis=[0, 0, 1])
antenna = Component(
    "antenna", "aluminium", place_triangles(build_cylinder_triangles(0.005, 0.2, capped=False), antenna_pose)
)

cubesat = join_components(
    [bus, deployed_panel("panel_plus_y", +1), deployed_panel("panel_minus_y", -1), antenna],
    source="3U CubeSat with two deployed panels (tutorial)",
)
print(cubesat.panel_count, "triangles")
print("components:", cubesat.component_names)
print("materials: ", cubesat.material_names)

1396 triangles
components: ('bus', 'panel_plus_y', 'panel_minus_y', 'antenna')
materials:  ('aluminium', 'solar_cell')


Coloured by component, each part can be hidden from the legend; coloured by material, the two panels share one colour. The panels face +x on one side and −x on the other: as two-sided plates, both faces are exposed.

In [16]:
plot_panel_set_3d(cubesat)

In [17]:
plot_panel_set_3d(cubesat, color_by="material", show_edges=False)

## 7. Checking that normals point outward (A3b3)

A triangle listed clockwise has a normal pointing into the spacecraft, which silently turns a ram face into a wake face. `join_components` now checks every component as it joins them:

- a **closed** component (a box, a capped cylinder) must be consistently wound with a positive enclosed volume, or joining raises an error;
- a **two-sided** open component (a deployed panel) faces both ways by construction;
- a **one-sided** open component is flagged with a `GeometryOrientationWarning` when most of its area faces the spacecraft's centre.

`summarise_geometry` shows the result per component; in a notebook it displays as a table, with rows needing attention shaded.

In [18]:
from leodragkit.aero.geometry_checks import check_component, summarise_geometry

summarise_geometry(cubesat)

component,triangles,area (m²),closed,consistent,two-sided,signed volume (m³),status
bus,780,0.156,yes,yes,no,0.0034,"ok: closed, outward"
panel_plus_y,300,0.06,no,–,yes,–,"ok: open, two-sided"
panel_minus_y,300,0.06,no,–,yes,–,"ok: open, two-sided"
antenna,16,0.006123,no,yes,no,–,"ok: open, one-sided"


A box with one triangle listed in the wrong order is caught, and the message names the triangles on either side of the broken edge:

In [19]:
broken_box_m = build_box_triangles(0.2, 0.2, 0.2)
broken_box_m[0] = broken_box_m[0][::-1]  # reverse one triangle's corners

try:
    check_component("broken box", broken_box_m)
except ValueError as error:
    print(error)

Component 'broken box': triangles 0 and 1 run along a shared edge in the same direction, so one of them is flipped: listed clockwise, not counter-clockwise, seen from outside.


Plotted with its normals, the flipped triangle's normal points into the box. (`single_component` builds a `PanelSet` directly, without the checks, so the broken box can be drawn.)

In [20]:
plot_panel_set_3d(single_component(broken_box_m, "broken box"), normal_length_m=0.08)

A one-sided panel typed facing the bus instead of away from it can't be proven wrong (it could be deliberate), so it is a warning rather than an error. The warning names the component and its panels; the summary row is shaded.

In [21]:
facing_bus_pose = make_pose(centre_m=[0.0, 0.25, 0.0], x_axis=[1, 0, 0], z_axis=[0, -1, 0])
one_sided_panel = Component(
    "panel_facing_bus", "solar_cell", place_triangles(build_plate_triangles(0.3, 0.1), facing_bus_pose)
)
with_warning = join_components([bus, one_sided_panel], source="bus with a one-sided panel facing it")
summarise_geometry(with_warning)

C:\Users\murph\AppData\Local\Temp\ipykernel_2616\978508741.py:5: GeometryOrientationWarning: bus with a one-sided panel facing it: component 'panel_facing_bus' is one-sided and 2 of its panels face the spacecraft's centre (panels 780, 781). If it should face outward, turn its pose's z axis round; if it is exposed on both sides, build it two-sided.
  with_warning = join_components([bus, one_sided_panel], source="bus with a one-sided panel facing it")


component,triangles,area (m²),closed,consistent,two-sided,signed volume (m³),status
bus,780,0.156,yes,yes,no,0.0034,"ok: closed, outward"
panel_facing_bus,2,0.03,no,yes,no,–,check: 2 panel(s) face the body


To make such warnings errors, for example in a production run:

```python
import warnings
warnings.simplefilter("error", GeometryOrientationWarning)
```

## 8. Placing parts: planes and rotations (A3c)

A pose fixes where a part sits (`centre_m`) and which way it faces. There are two ways to give the orientation:

- **Axes:** `z_axis` is where the part's own +z (a plate's front face) points, and `x_axis` is where its own x (a plate's first side) runs. They must be perpendicular; the third axis is y = z × x.
- **Turn:** turn by `rotation_angle_deg` about `rotation_axis`, an axis of the part's own frame, by the right-hand rule: with your right thumb along the axis, a positive angle turns the way your fingers curl. So +30° about +x turns +z towards **−y**. Without axes the part starts unrotated, so its own axes are the body axes.
- **Both:** place by axes, then turn about the part's own axis, for example tilting a panel about its own long side.

Below, the same 0.3 × 0.1 m one-sided plate is placed eight ways, side by side along body x. Each line from a plate is its outward normal. The guide `docs/guides/describing_a_spacecraft.md` explains each case.

In [22]:
from leodragkit.aero.geometry_builder import make_pose_from_rotation, turn_pose_about_own_axis

placements = {  # name: pose
    "xy plane, facing +z": make_pose([0.0, 0, 0], x_axis=[1, 0, 0], z_axis=[0, 0, 1]),
    "xz plane, facing +y": make_pose([0.6, 0, 0], x_axis=[1, 0, 0], z_axis=[0, 1, 0]),
    "yz plane, facing +x": make_pose([1.2, 0, 0], x_axis=[0, 1, 0], z_axis=[1, 0, 0]),
    "yz plane, facing -x": make_pose([1.8, 0, 0], x_axis=[0, 1, 0], z_axis=[-1, 0, 0]),
    "+30 deg about x (rotation form)": make_pose_from_rotation(
        [2.4, 0, 0], rotation_axis=[1, 0, 0], rotation_angle_deg=30
    ),
    "-30 deg about x (axes form)": make_pose([3.0, 0, 0], x_axis=[1, 0, 0], z_axis=[0, 0.5, 0.866]),
    "facing (1, 1, 1)": make_pose([3.6, 0, 0], x_axis=[0.7071, -0.7071, 0], z_axis=[0.5774, 0.5774, 0.5774]),
    "facing +x, then 20 deg about own x": turn_pose_about_own_axis(
        make_pose([4.2, 0, 0], x_axis=[0, 1, 0], z_axis=[1, 0, 0]),
        rotation_axis=[1, 0, 0],
        rotation_angle_deg=20,
    ),
}
for name, pose in placements.items():
    print(f"{name:34s} normal (z axis) = {np.round(pose.z_axis, 3)}   own x along {np.round(pose.x_axis, 3)}")

xy plane, facing +z                normal (z axis) = [0. 0. 1.]   own x along [1. 0. 0.]
xz plane, facing +y                normal (z axis) = [0. 1. 0.]   own x along [1. 0. 0.]
yz plane, facing +x                normal (z axis) = [1. 0. 0.]   own x along [0. 1. 0.]
yz plane, facing -x                normal (z axis) = [-1.  0.  0.]   own x along [0. 1. 0.]
+30 deg about x (rotation form)    normal (z axis) = [ 0.    -0.5    0.866]   own x along [1. 0. 0.]
-30 deg about x (axes form)        normal (z axis) = [0.    0.5   0.866]   own x along [1. 0. 0.]
facing (1, 1, 1)                   normal (z axis) = [0.577 0.577 0.577]   own x along [ 0.707 -0.707  0.   ]
facing +x, then 20 deg about own x normal (z axis) = [ 0.94   0.    -0.342]   own x along [0. 1. 0.]


To draw them together without the orientation checks (several one-sided plates facing every which way would rightly be flagged), the triangles go straight into one `PanelSet`, one component per placement:

In [ ]:
plate_m = build_plate_triangles(0.3, 0.1)
gallery_triangles = np.concatenate([place_triangles(plate_m, pose) for pose in placements.values()])
gallery = panel_set_from_triangles(
    gallery_triangles,
    component_ids=np.repeat(np.arange(len(placements)), len(plate_m)),
    material_ids=[0] * len(gallery_triangles),
    component_names=tuple(placements),
    material_names=("solar_cell",),
    source="placement gallery",
)
pfig = plot_panel_set_3d(gallery, normal_length_m=0.2)
pfig.update_traces(visible="legendonly", selector=dict(name="body y"))
pfig.update_traces(visible="legendonly", selector=dict(name="body z"))
pfig.show()


## 9. A spacecraft from a YAML file (A3c)

Most spacecraft are described in a short YAML file rather than in Python. `read_spacecraft_description_file` reads it (from `leodragkit.io`), and `build_spacecraft` checks every key, builds each component at its pose, joins them (running the orientation checks) and attaches the materials. This is the same 3U CubeSat as section 6, from `config/geometry/cubesat_3u_example.yaml`.

In [24]:
from pathlib import Path

from leodragkit.aero.spacecraft import build_spacecraft
from leodragkit.io.geometry_files import read_spacecraft_description_file

example_path = Path("..") / "config" / "geometry" / "cubesat_3u_example.yaml"
description = read_spacecraft_description_file(example_path)
cubesat_from_file = build_spacecraft(description, source=example_path.name)
print(
    cubesat_from_file.name,
    "|",
    cubesat_from_file.mass_kg,
    "kg |",
    cubesat_from_file.panel_set.panel_count,
    "triangles",
)
for material in cubesat_from_file.materials:
    wall, alpha = material.wall_temperature_kelvin, material.energy_accommodation
    print(f"  {material.name}: wall {wall} K, accommodation {alpha}")

3U CubeSat with deployed panels (example) | 4.0 kg | 1700 triangles
  aluminium: wall 300.0 K, accommodation 0.9
  solar_cell: wall 300.0 K, accommodation 0.9


In [25]:
plot_panel_set_3d(cubesat_from_file.panel_set)

In [26]:
summarise_geometry(cubesat_from_file.panel_set)

component,triangles,area (m²),closed,consistent,two-sided,signed volume (m³),status
bus,780,0.156,yes,yes,no,0.0034,"ok: closed, outward"
panel_plus_y,300,0.06,no,–,yes,–,"ok: open, two-sided"
panel_minus_y,300,0.06,no,–,yes,–,"ok: open, two-sided"
antenna,320,0.006123,no,yes,no,–,"ok: open, one-sided"


A mistake in the file is reported with its place in the description. Here `size_m` is mistyped as `sizes_m` in the second component:

In [27]:
import copy

mistyped = copy.deepcopy(description)
mistyped["components"][1]["sizes_m"] = mistyped["components"][1].pop("size_m")
try:
    build_spacecraft(mistyped, source="mistyped example")
except ValueError as error:
    print(error)

components[1] (panel_plus_y) is missing ['size_m'] and has unknown keys ['sizes_m']; allowed: ['material', 'max_edge_length_m', 'name', 'pose', 'shape', 'size_m', 'two_sided'].


A material defined but used by no component is left out, with a warning in case it was meant to be used:

In [28]:
with_spare_material = copy.deepcopy(description)
with_spare_material["materials"]["gold_foil"] = {
    "wall_temperature_kelvin": 300.0,
    "energy_accommodation": 0.8,
}
_ = build_spacecraft(with_spare_material, source="example with a spare material")

C:\Users\murph\AppData\Local\Temp\ipykernel_2616\983870762.py:6: SpacecraftDescriptionWarning: example with a spare material: materials ['gold_foil'] are defined but used by no component, so they are left out. If a component should use one, check its 'material' spelling.
  _ = build_spacecraft(with_spare_material, source="example with a spare material")


## Saving a figure

Plot functions never write files. To keep a figure, save it yourself, for example into the output directory from your main config:

```python
figure.write_html("cube.html")
```

## Coming next

- **A3d**: macro-model tables and STL or OBJ meshes from files.
- **A4**: the incoming flow drawn on the spacecraft (plot level 2).